# Resultados finales y galería de errores
Los intervalos describen la variación entre semillas sobre el mismo TEST. No estiman la incertidumbre de muestreo de la población de tapas.

In [ ]:
from pathlib import Path
from IPython import get_ipython

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)


def load_notebook(name):
    path = ROOT / "notebooks" / "shared" / (name + ".ipynb")
    get_ipython().run_line_magic("run", f'-i "{path.as_posix()}"')


load_notebook("workspace")

In [ ]:
load_notebook("experiment")
from scipy.stats import t
import matplotlib.pyplot as plt

_, dataset_hash = cap_rows()
suite = RUNS.parent / "final" / dataset_hash
plan = json.loads((suite / "plan.json").read_text())
results = []
for name in plan["runs"]:
    record = json.loads((RUNS / name / "record.json").read_text(encoding="utf-8"))
    metrics = json.loads((suite / name / "metrics.json").read_text())
    results.append(
        dict(
            model=record["config"]["model"],
            seed=record["seed"],
            run=name,
            fit_seconds=record["fit_seconds"],
            peak_reserved_gib=record["peak_reserved_gib"],
            **metrics,
        )
    )
table = pd.DataFrame(results)
table.to_csv(suite / "runs.csv", index=False)
columns = [
    "image_auroc",
    "image_ap",
    "pixel_auroc",
    "pixel_ap",
    "au_pro",
    "precision",
    "recall",
    "f1",
    "latency_p50_ms",
    "latency_p95_ms",
    "images_per_second",
    "fit_seconds",
    "peak_reserved_gib",
]
summary = (
    table.melt(id_vars=["model", "seed"], value_vars=columns)
    .groupby(["model", "variable"])
    .value.agg(["mean", "std", "count"])
)
margin = t.ppf(0.975, summary["count"] - 1) * summary["std"] / np.sqrt(summary["count"])
summary.assign(
    ci_low=summary["mean"] - margin, ci_high=summary["mean"] + margin
).to_csv(suite / "summary.csv")
for filename in ("runs.csv", "summary.csv"):
    export_study_file(suite, filename)
figure, axis = plt.subplots(figsize=(8, 4))
table.boxplot(column="image_auroc", by="model", ax=axis)
axis.set(xlabel="Arquitectura", ylabel="AUROC por imagen", title="Variación entre semillas en TEST")
figure.suptitle("")
figure.tight_layout()
figure.savefig(suite / "image-auroc.png")
plt.close(figure)
export_study_file(suite, "image-auroc.png")
display(summary)

In [ ]:
for row in table.itertuples():
    folder = suite / row.run
    scores = pd.read_csv(folder / "predictions.csv")
    wrong = scores[(scores.score >= row.threshold) != scores.label.astype(bool)]
    for index, sample in wrong.head(4).iterrows():
        with (
            Image.open(sample.path) as source,
            np.load(folder / f"{index:06d}.npz") as maps,
        ):
            figure, panels = plt.subplots(1, 3, figsize=(9, 3))
            for panel, content, title in zip(
                panels,
                [source, maps["mask"], maps["score"]],
                ["Captura", "Máscara central", "Mapa central"],
            ):
                panel.imshow(content)
                panel.set_title(title)
                panel.axis("off")
            figure.suptitle(
                f"{row.model} · semilla {row.seed} · etiqueta {sample.label} · puntaje {sample.score:.3f}"
            )
            figure.tight_layout()
            figure.savefig(folder / f"error-{index}.png")
            plt.close(figure)